# Setup & Imports

In [1]:
import torch
import tiktoken
from pathlib import Path
from copy import deepcopy
import pandas as pd
from torch.utils.data import DataLoader

from model import GPTModel, GPTClassifier
from config import GPT_CONFIG_SMALL, GPT_CONFIG_MEDIUM, GPT_CONFIG_LARGE, GPT_CONFIG_XLARGE
from generation import generate_text, classify_text
from weights import load_pretrained_gpt2, load_classifier
from checkpoint import save_model, save_checkpoint
from data import (
    create_dataloaders, split_text_data,
    download_and_unzip_spam_data, create_balanced_dataset,
    random_split, SpamDataset
)
from engine import train_model_simple, train_classifier_simple, calc_accuracy_loader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

tokenizer = tiktoken.get_encoding('gpt2')
print("Tokenizer loaded!")

Using device: cpu
Tokenizer loaded!


In [2]:
CONFIGS = {
    'small': GPT_CONFIG_SMALL,
    'medium': GPT_CONFIG_MEDIUM,
    'large': GPT_CONFIG_LARGE,
    'xlarge': GPT_CONFIG_XLARGE
}

GPT2_SIZE_MAP = {
    'small': '124M',
    'medium': '355M',
    'large': '774M',
    'xlarge': '1558M'
}

# Inference 

## Generation

In [3]:
generation_model, gen_config = load_pretrained_gpt2('small')

Downloading/loading GPT-2 124M...
File already exists and is up-to-date: gpt2\124M\checkpoint
File already exists and is up-to-date: gpt2\124M\encoder.json
File already exists and is up-to-date: gpt2\124M\hparams.json
File already exists and is up-to-date: gpt2\124M\model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2\124M\model.ckpt.index
File already exists and is up-to-date: gpt2\124M\model.ckpt.meta


In [5]:
prompt = "The future of artificial intelligence is"

max_new_tokens = 50  
temperature = 0.7     
top_k = 50            

print("="*60)
print("TEXT GENERATION")
print("="*60)
print(f"Prompt: {prompt}\n")
print("Generated Text:")
print("-"*60)
result = generate_text(prompt, generation_model, tokenizer, device,max_new_tokens, temperature, top_k)
print(result)
print("-"*60)

TEXT GENERATION
Prompt: The future of artificial intelligence is

Generated Text:
------------------------------------------------------------
The future of artificial intelligence is not in the hands of the government. The future of AI is in the hands of the private sector.

As such, the government should work with the private sector to develop a sustainable future for the development of AI. This could include developing new
------------------------------------------------------------


## Spam Classification 

In [14]:
classifier_model, clf_config = load_classifier('models/gpt_classifier_frozen.pt', device=device)
MAX_LENGTH = clf_config.get('max_length', 120)

Loading classifier from models/gpt_classifier_frozen.pt...
✓ Config loaded from checkpoint
✓ Classifier loaded!


In [15]:
text_to_classify = "Congratulations! You've won a free iPhone. Click here to claim your prize now!"

print("="*60)
print("SPAM CLASSIFICATION")
print("="*60)
print(f"Input: {text_to_classify}\n")

result = classify_text(text_to_classify, classifier_model, tokenizer, device, max_length=MAX_LENGTH, return_probs=True)

print("Result:")
print("-"*60)
print(f"Prediction: {result['prediction']}")
print(f"Confidence: {result['confidence']:.2%}")

SPAM CLASSIFICATION
Input: Congratulations! You've won a free iPhone. Click here to claim your prize now!

Result:
------------------------------------------------------------
Prediction: NOT SPAM
Confidence: 85.80%


In [16]:
test_texts = [
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT: Your account has been compromised! Click here immediately!",
    "Thanks for the report. I'll review it and get back to you.",
    "You have been selected for a $1000 Walmart gift card! Call now!",
    "Can you pick up some milk on your way home?"
]

print("="*60)
print("BATCH CLASSIFICATION")
print("="*60)

for i, text in enumerate(test_texts, 1):
    result = classify_text(text, classifier_model, tokenizer, device, MAX_LENGTH,return_probs=True)
    print(f"\n{i}. {text[:50]}..." if len(text) > 50 else f"\n{i}. {text}")
    print(f"   → {result['prediction']} ({result['confidence']:.1%} confident)")

BATCH CLASSIFICATION

1. Hey, are we still meeting for lunch tomorrow?
   → NOT SPAM (93.5% confident)

2. URGENT: Your account has been compromised! Click h...
   → NOT SPAM (88.6% confident)

3. Thanks for the report. I'll review it and get back...
   → NOT SPAM (91.1% confident)

4. You have been selected for a $1000 Walmart gift ca...
   → NOT SPAM (76.3% confident)

5. Can you pick up some milk on your way home?
   → NOT SPAM (95.2% confident)
